# TP — Réseau de Neurones XOR avec NumPy

**Objectif :** Implémenter from scratch un réseau de neurones multicouche (MLP) capable de résoudre le problème XOR, qui n'est **pas linéairement séparable**.

---

## 1. Rappel théorique

### Le problème XOR

| x1 | x2 | XOR |
|----|----|-----|
|  0 |  0 |  0  |
|  0 |  1 |  1  |
|  1 |  0 |  1  |
|  1 |  1 |  0  |

Un perceptron simple **ne peut pas** apprendre XOR. Il faut au minimum :
- une **couche cachée** avec une **fonction d'activation non-linéaire**
- une couche de sortie

### Architecture choisie

```
Entrée (2) → Couche cachée (4, Sigmoid) → Sortie (1, Sigmoid)
```

### Algorithme de rétropropagation (Backpropagation)

1. **Forward pass** : calculer la prédiction
2. **Calcul de la perte** : Binary Cross-Entropy ou MSE
3. **Backward pass** : calculer les gradients via la règle de la chaîne
4. **Mise à jour des poids** : descente de gradient

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

np.random.seed(42)  # reproductibilité

## 3. Données XOR

In [ ]:
# Entrées : 4 exemples, 2 features
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

# Sorties attendues (XOR)
y = np.array([[0],
              [1],
              [1],
              [0]], dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nDonnées XOR :")
for xi, yi in zip(X, y):
    print(f"  {xi[0]:.0f} XOR {xi[1]:.0f} = {yi[0]:.0f}")

## 4. Fonctions d'activation

In [ ]:
def sigmoid(z):
    """Sigmoid : σ(z) = 1 / (1 + e^(-z))"""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z):
    """Dérivée de sigmoid : σ'(z) = σ(z) * (1 - σ(z))"""
    s = sigmoid(z)
    return s * (1 - s)

def tanh_act(z):
    return np.tanh(z)

def tanh_prime(z):
    return 1 - np.tanh(z)**2

def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return (z > 0).astype(float)

# Visualisation des fonctions et de leurs dérivées
z = np.linspace(-5, 5, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fns = [
    ("Sigmoid", sigmoid, sigmoid_prime),
    ("Tanh",    tanh_act, tanh_prime),
    ("ReLU",    relu,     relu_prime),
]

for ax, (name, fn, dfn) in zip(axes, fns):
    ax.plot(z, fn(z),  label=name, linewidth=2)
    ax.plot(z, dfn(z), label=f"{name}'", linewidth=2, linestyle='--')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Fonctions d'activation et leurs dérivées", fontsize=14)
plt.tight_layout()
plt.savefig("activation_functions.png", dpi=150)
plt.show()

## 5. Initialisation du réseau

Architecture : **2 → 4 → 1**

- `W1` : poids couche 1, shape **(4, 2)**
- `b1` : biais couche 1, shape **(4, 1)**
- `W2` : poids couche 2, shape **(1, 4)**
- `b2` : biais couche 2, shape **(1, 1)**

In [ ]:
def init_weights(n_input=2, n_hidden=4, n_output=1):
    """
    Initialisation Xavier/Glorot : réduit le problème
    de vanishing/exploding gradients.
    """
    W1 = np.random.randn(n_hidden, n_input) * np.sqrt(2.0 / n_input)
    b1 = np.zeros((n_hidden, 1))
    W2 = np.random.randn(n_output, n_hidden) * np.sqrt(2.0 / n_hidden)
    b2 = np.zeros((n_output, 1))
    return W1, b1, W2, b2

W1, b1, W2, b2 = init_weights()
print("Formes des paramètres :")
print(f"  W1 : {W1.shape}")
print(f"  b1 : {b1.shape}")
print(f"  W2 : {W2.shape}")
print(f"  b2 : {b2.shape}")

## 6. Forward Pass

In [ ]:
def forward(X, W1, b1, W2, b2):
    """
    Propagation avant.
    X : (n_samples, n_features)
    Retourne : prédiction et valeurs intermédiaires pour le backprop.
    """
    Xt = X.T                      # (2, 4)

    Z1 = W1 @ Xt + b1             # (4, 4)
    A1 = sigmoid(Z1)              # (4, 4)

    Z2 = W2 @ A1 + b2             # (1, 4)
    A2 = sigmoid(Z2)              # (1, 4)

    cache = (Z1, A1, Z2, A2, Xt)
    return A2, cache

# Test
A2, cache = forward(X, W1, b1, W2, b2)
print("Prédictions initiales (avant entraînement) :")
print(A2.T.round(4))

## 7. Fonction de perte — Binary Cross-Entropy

$$\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

In [ ]:
def compute_loss(A2, y):
    """
    Binary Cross-Entropy Loss.
    A2 : prédictions (1, m)
    y  : vraies valeurs (m, 1)
    """
    m = y.shape[0]
    eps = 1e-8  # évite log(0)
    yT = y.T   # (1, m)
    loss = -np.mean(yT * np.log(A2 + eps) + (1 - yT) * np.log(1 - A2 + eps))
    return loss

loss = compute_loss(A2, y)
print(f"Perte initiale : {loss:.4f}")

## 8. Backward Pass — Rétropropagation

Gradients calculés via la règle de la chaîne :

$$\delta_2 = A_2 - y$$
$$\nabla W_2 = \frac{1}{m} \delta_2 \cdot A_1^T$$
$$\delta_1 = (W_2^T \cdot \delta_2) \odot \sigma'(Z_1)$$
$$\nabla W_1 = \frac{1}{m} \delta_1 \cdot X^T$$

In [ ]:
def backward(A2, y, cache, W2):
    """
    Rétropropagation — calcule les gradients.
    """
    Z1, A1, Z2, _, Xt = cache
    m = y.shape[0]
    yT = y.T  # (1, m)

    # Couche de sortie
    dZ2 = A2 - yT                          # (1, m)
    dW2 = (dZ2 @ A1.T) / m                 # (1, 4)
    db2 = np.mean(dZ2, axis=1, keepdims=True)  # (1, 1)

    # Couche cachée
    dA1 = W2.T @ dZ2                       # (4, m)
    dZ1 = dA1 * sigmoid_prime(Z1)          # (4, m)
    dW1 = (dZ1 @ Xt.T) / m                 # (4, 2)
    db1 = np.mean(dZ1, axis=1, keepdims=True)  # (4, 1)

    return dW1, db1, dW2, db2

dW1, db1_, dW2, db2_ = backward(A2, y, cache, W2)
print("Gradients calculés :")
print(f"  dW1 shape : {dW1.shape}")
print(f"  dW2 shape : {dW2.shape}")

## 9. Entraînement complet

In [ ]:
def train(X, y, n_hidden=4, lr=0.5, epochs=10000, print_every=1000):
    """
    Entraînement du réseau XOR.
    
    Paramètres
    ----------
    X         : données d'entrée (4, 2)
    y         : étiquettes cibles (4, 1)
    n_hidden  : nb de neurones cachés
    lr        : taux d'apprentissage (learning rate)
    epochs    : nombre d'itérations
    """
    W1, b1, W2, b2 = init_weights(n_input=2, n_hidden=n_hidden, n_output=1)
    history = []

    for epoch in range(epochs + 1):
        # Forward
        A2, cache = forward(X, W1, b1, W2, b2)

        # Perte
        loss = compute_loss(A2, y)
        history.append(loss)

        # Backward
        dW1, db1_grad, dW2, db2_grad = backward(A2, y, cache, W2)

        # Mise à jour
        W1 -= lr * dW1
        b1 -= lr * db1_grad
        W2 -= lr * dW2
        b2 -= lr * db2_grad

        if epoch % print_every == 0:
            preds = (A2.T > 0.5).astype(int).flatten()
            acc = np.mean(preds == y.flatten().astype(int)) * 100
            print(f"Époque {epoch:6d} | Perte : {loss:.6f} | Précision : {acc:.1f}%")

    return W1, b1, W2, b2, history


print("Démarrage de l'entraînement...\n")
W1, b1, W2, b2, history = train(X, y, n_hidden=4, lr=1.0, epochs=10000)

## 10. Courbe d'apprentissage

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history, color='steelblue', linewidth=1.5)
plt.xlabel("Époque")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Courbe d'apprentissage — Réseau XOR")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve.png", dpi=150)
plt.show()
print(f"\nPerte finale : {history[-1]:.6f}")

## 11. Évaluation finale

In [ ]:
A2_final, _ = forward(X, W1, b1, W2, b2)
preds = (A2_final.T > 0.5).astype(int).flatten()

print("Résultats finaux :\n")
print(f"{'x1':>4} {'x2':>4} {'Attendu':>9} {'Prédit':>7} {'Confiance':>10}")
print("-" * 40)
for i in range(4):
    conf = A2_final[0, i]
    expected = int(y[i, 0])
    predicted = preds[i]
    ok = "✓" if expected == predicted else "✗"
    print(f"{X[i,0]:>4.0f} {X[i,1]:>4.0f} {expected:>9d} {predicted:>7d} {conf:>9.4f}  {ok}")

accuracy = np.mean(preds == y.flatten().astype(int)) * 100
print(f"\nPrécision : {accuracy:.1f}%")

## 12. Visualisation de la frontière de décision

In [ ]:
def plot_decision_boundary(W1, b1, W2, b2, resolution=300):
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, resolution),
                         np.linspace(-0.5, 1.5, resolution))
    grid = np.c_[xx.ravel(), yy.ravel()]
    A2_grid, _ = forward(grid, W1, b1, W2, b2)
    Z = A2_grid.reshape(resolution, resolution)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Probabilités
    im = axes[0].contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.8)
    plt.colorbar(im, ax=axes[0], label='P(sortie=1)')
    axes[0].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    colors = ['red' if yi == 0 else 'green' for yi in y.flatten()]
    axes[0].scatter(X[:, 0], X[:, 1], c=colors, s=200,
                    edgecolors='black', zorder=5)
    for i, (xi, yi) in enumerate(zip(X, y)):
        axes[0].annotate(f"({xi[0]:.0f},{xi[1]:.0f})→{yi[0]:.0f}",
                         (xi[0] + 0.05, xi[1] + 0.05), fontsize=9)
    axes[0].set_title("Frontière de décision (probabilités)")
    axes[0].set_xlabel("x1")
    axes[0].set_ylabel("x2")

    # Décision binaire
    Z_bin = (Z > 0.5).astype(int)
    axes[1].contourf(xx, yy, Z_bin, cmap='RdYlGn', alpha=0.4)
    axes[1].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    axes[1].scatter(X[:, 0], X[:, 1], c=colors, s=200,
                    edgecolors='black', zorder=5)
    axes[1].set_title("Décision binaire (seuil = 0.5)")
    axes[1].set_xlabel("x1")
    axes[1].set_ylabel("x2")

    plt.suptitle("Réseau de neurones XOR — Visualisation", fontsize=14)
    plt.tight_layout()
    plt.savefig("decision_boundary.png", dpi=150)
    plt.show()

plot_decision_boundary(W1, b1, W2, b2)

## 13. Comparaison : impact du nombre de neurones cachés

In [ ]:
hidden_sizes = [2, 4, 8, 16]
results = {}

for n_h in hidden_sizes:
    np.random.seed(42)
    W1_, b1_, W2_, b2_, hist = train(X, y, n_hidden=n_h, lr=1.0,
                                      epochs=5000, print_every=99999)
    A2_, _ = forward(X, W1_, b1_, W2_, b2_)
    preds_ = (A2_.T > 0.5).astype(int).flatten()
    acc = np.mean(preds_ == y.flatten().astype(int)) * 100
    results[n_h] = {"history": hist, "accuracy": acc,
                    "weights": (W1_, b1_, W2_, b2_)}
    print(f"n_hidden={n_h:2d} | Perte finale={hist[-1]:.5f} | Précision={acc:.1f}%")

# Courbes de perte
plt.figure(figsize=(10, 4))
for n_h, res in results.items():
    plt.plot(res["history"], label=f"n_hidden={n_h} (acc={res['accuracy']:.0f}%)")
plt.xlabel("Époque")
plt.ylabel("Loss")
plt.title("Impact du nombre de neurones cachés")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("hidden_size_comparison.png", dpi=150)
plt.show()

## 14. Comparaison : impact du taux d'apprentissage

In [ ]:
learning_rates = [0.01, 0.1, 0.5, 1.0, 2.0]
lr_results = {}

plt.figure(figsize=(11, 5))
for lr in learning_rates:
    np.random.seed(42)
    _, _, _, _, hist = train(X, y, n_hidden=4, lr=lr,
                              epochs=5000, print_every=99999)
    plt.plot(hist, label=f"lr={lr}")
    lr_results[lr] = hist

plt.xlabel("Époque")
plt.ylabel("Loss")
plt.title("Impact du taux d'apprentissage")
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 1.2)
plt.tight_layout()
plt.savefig("lr_comparison.png", dpi=150)
plt.show()

## 15. Momentum (SGD avec momentum)

Le momentum accélère la convergence en accumulant une vitesse dans les directions de descente stables :

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot \nabla W$$
$$W \leftarrow W - \alpha \cdot v_t$$

In [ ]:
def train_with_momentum(X, y, n_hidden=4, lr=0.5, beta=0.9, epochs=5000):
    W1, b1, W2, b2 = init_weights(n_input=2, n_hidden=n_hidden)
    
    # Vitesses (momentum)
    vW1 = np.zeros_like(W1)
    vb1 = np.zeros_like(b1)
    vW2 = np.zeros_like(W2)
    vb2 = np.zeros_like(b2)
    history = []

    for epoch in range(epochs + 1):
        A2, cache = forward(X, W1, b1, W2, b2)
        loss = compute_loss(A2, y)
        history.append(loss)
        dW1, db1g, dW2, db2g = backward(A2, y, cache, W2)

        # Mise à jour avec momentum
        vW1 = beta * vW1 + (1 - beta) * dW1
        vb1 = beta * vb1 + (1 - beta) * db1g
        vW2 = beta * vW2 + (1 - beta) * dW2
        vb2 = beta * vb2 + (1 - beta) * db2g

        W1 -= lr * vW1
        b1 -= lr * vb1
        W2 -= lr * vW2
        b2 -= lr * vb2

    return W1, b1, W2, b2, history

# Comparaison SGD vs Momentum
np.random.seed(42)
_, _, _, _, hist_sgd  = train(X, y, lr=0.5, epochs=5000, print_every=99999)
np.random.seed(42)
_, _, _, _, hist_mom  = train_with_momentum(X, y, lr=0.5, beta=0.9, epochs=5000)

plt.figure(figsize=(9, 4))
plt.plot(hist_sgd, label="SGD simple", linewidth=1.5)
plt.plot(hist_mom, label="SGD + Momentum (β=0.9)", linewidth=1.5, linestyle='--')
plt.xlabel("Époque")
plt.ylabel("Loss")
plt.title("SGD vs SGD avec Momentum")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("momentum_comparison.png", dpi=150)
plt.show()

## 16. Exercices à compléter

### Exercice 1 — Changer la fonction d'activation
Modifiez le réseau pour utiliser `tanh` dans la couche cachée à la place de `sigmoid`. Quelle différence observez-vous sur la convergence ?

In [ ]:
def forward_tanh(X, W1, b1, W2, b2):
    """
    TODO : Implémenter le forward pass avec tanh dans la couche cachée.
    Couche cachée : tanh
    Couche de sortie : sigmoid (inchangée)
    """
    # VOTRE CODE ICI
    pass

def backward_tanh(A2, y, cache, W2):
    """
    TODO : Implémenter le backward pass correspondant (dérivée de tanh).
    """
    # VOTRE CODE ICI
    pass

### Exercice 2 — Réseau à 3 couches
Ajoutez une deuxième couche cachée : **2 → 4 → 4 → 1**. Implémentez le forward et le backward pass complets.

In [ ]:
def init_3layers(n_input=2, n_h1=4, n_h2=4, n_output=1):
    """
    TODO : Initialiser les poids d'un réseau à 3 couches.
    """
    # VOTRE CODE ICI
    pass

def forward_3layers(X, W1, b1, W2, b2, W3, b3):
    """
    TODO : Forward pass pour le réseau 2→4→4→1.
    """
    # VOTRE CODE ICI
    pass

def backward_3layers(A3, y, cache, W2, W3):
    """
    TODO : Backward pass pour le réseau 2→4→4→1.
    """
    # VOTRE CODE ICI
    pass

### Exercice 3 — Régularisation L2
Implémentez la régularisation L2 (weight decay) dans la fonction de perte :

$$\mathcal{L}_{reg} = \mathcal{L} + \frac{\lambda}{2m} \left( \|W_1\|^2_F + \|W_2\|^2_F \right)$$

Que se passe-t-il avec `λ = 0.01` vs `λ = 1.0` ?

In [ ]:
def compute_loss_l2(A2, y, W1, W2, lambd=0.01):
    """
    TODO : Binary Cross-Entropy + régularisation L2.
    """
    # VOTRE CODE ICI
    pass

## 17. Résumé

| Composant | Description |
|-----------|-------------|
| Architecture | 2 → 4 → 1 |
| Activation cachée | Sigmoid |
| Activation sortie | Sigmoid |
| Perte | Binary Cross-Entropy |
| Optimiseur | SGD / SGD + Momentum |
| Initialisation | Xavier/Glorot |

**Points clés retenus :**
1. XOR est non linéairement séparable → une couche cachée est nécessaire
2. L'initialisation des poids influence fortement la convergence
3. Un taux d'apprentissage trop élevé peut diverger, trop faible → convergence lente
4. Le momentum accélère la convergence